# 05_data_train

对应 `index.md` 第 5 阶段：Dataset / DataLoader、loss、optim、一步训练。

笔记本在 `codes/pytorch/05_data_train/qa.ipynb`。需要读写文件时，工作空间就是这个目录。

先运行下一格，得到 `ROOT`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。


In [1]:
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "05_data_train" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "pytorch" / "05_data_train"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

ROOT = lab_root()
ROOT


PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/pytorch/05_data_train')

## 1. 自定义 Dataset

写一个继承 `Dataset` 的类 `TinySet`：

- `__init__` 接收特征张量 `features` 和标签张量 `labels` 并保存
- `__len__` 返回样本数
- `__getitem__` 按索引返回 `(features[i], labels[i])`

用已有的 `X`、`y` 实例化，打印 `len(ds)`，再打印 `ds[1]`。


In [2]:
import torch
from torch.utils.data import Dataset

X = torch.arange(12, dtype=torch.float32).reshape(6, 2)
y = torch.arange(6)

# 作答

import torch
from torch.utils.data import Dataset

class TinySet(Dataset):
    def __init__(self, features : torch.Tensor, labels : torch.Tensor) -> None:
        assert len(features) == len(labels), "features 和 labels 的样本数必须一致"
        self.features = features
        self.labels = labels

    def __len__(self) -> int:
        return len(self.features)

    def __getitem__(self, idx : int) -> tuple[torch.Tensor, torch.Tensor]:
        return (self.features[idx], self.labels[idx])

ds = TinySet(X, y)

print(len(ds))
print(ds[1])
#评阅
# 对。__init__ / __len__ / __getitem__ 齐全，len=6，ds[1] 也对。

#参考答案
# class TinySet(Dataset):
#     def __init__(self, features, labels):
#         self.features, self.labels = features, labels
#     def __len__(self):
#         return len(self.features)
#     def __getitem__(self, i):
#         return self.features[i], self.labels[i]
# ds = TinySet(X, y)
# print(len(ds), ds[1])

6
(tensor([2., 3.]), tensor(1))


## 2. DataLoader 取一个 batch

用已有的 `ds` 建 `DataLoader`：`batch_size=2`，`shuffle=False`。

取出**第一个** batch，分别打印特征和标签的 `shape`。


In [3]:
import torch
from torch.utils.data import DataLoader, TensorDataset

ds = TensorDataset(
    torch.arange(12, dtype=torch.float32).reshape(6, 2),
    torch.arange(6),
)

# 作答

import torch
from torch.utils.data import DataLoader

loader = DataLoader(ds, batch_size=2, shuffle=False)

xb, yb = next(iter(loader))

print(xb.shape, yb.shape)
#评阅
# 对。batch_size=2、shuffle=False，首个 batch shape 为 [2, 2] 与 [2]。

#参考答案
# loader = DataLoader(ds, batch_size=2, shuffle=False)
# xb, yb = next(iter(loader))
# print(xb.shape, yb.shape)

torch.Size([2, 2]) torch.Size([2])


## 3. 算一次 loss

已有模型输出 `pred` 和标签 `target`。用 `nn.MSELoss` 算损失，打印这个标量 loss（可用 `.item()`）。


In [5]:
import torch
from torch import nn

pred = torch.tensor([1.0, 2.0, 3.0])
target = torch.tensor([1.0, 1.0, 5.0])

# 作答

import torch
import torch.nn as nn

loss_fn = nn.MSELoss()
loss = loss_fn(pred, target)
print(loss.item())
#评阅
# 对。MSE = ((0)^2+(1)^2+(-2)^2)/3 ≈ 1.6667。

#参考答案
# print(nn.MSELoss()(pred, target).item())


1.6666666269302368


## 4. 挂上优化器

已有 `net`。用 `torch.optim.SGD` 建优化器：参数来自 `net.parameters()`，学习率 `0.1`。

打印优化器里参数组的学习率（`optimizer.param_groups[0]["lr"]`）。


In [6]:
import torch
from torch import nn

net = nn.Linear(3, 1)

# 作答

import torch
import torch.nn as nn

opt = torch.optim.SGD(net.parameters(), lr=0.1)

print(opt.param_groups[0]["lr"])
#评阅
# 对。SGD 挂上 net.parameters()，lr=0.1。

#参考答案
# optimizer = torch.optim.SGD(net.parameters(), lr=0.1)
# print(optimizer.param_groups[0]["lr"])

0.1


## 5. 完整一步训练

对已有的 `net`、`opt`、`loss_fn`、`xb`、`yb`，完成**一步**训练：

1. 梯度清零
2. 前向得到预测
3. 算 loss
4. `backward`
5. `opt.step()`

打印这一步的 loss（标量）。


In [7]:
import torch
from torch import nn

torch.manual_seed(0)
net = nn.Linear(2, 1)
opt = torch.optim.SGD(net.parameters(), lr=0.1)
loss_fn = nn.MSELoss()
xb = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
yb = torch.tensor([[1.0], [2.0]])

# 作答

import torch
import torch.nn as nn

opt.zero_grad()
pred = net(xb)
loss = loss_fn(pred, yb)
loss.backward()
opt.step()

print(loss.item())
#评阅
# 对。清零 → 前向 → loss → backward → step，顺序完整。

#参考答案
# opt.zero_grad()
# loss = loss_fn(net(xb), yb)
# loss.backward()
# opt.step()
# print(loss.item())



0.9271299839019775
